In [ ]:
# Text Preprocessing

# This notebook applies the fixed text-normalisation procedure used by both the classical and transformer models.
# The same preprocessing is applied to all domains and partitions.

In [1]:
import pandas as pd

from src.config import (
    RAW_TEXT_COL,
    TEXT_COL,
    RU_PREPROCESSED_FILE,
    NONRU_PREPROCESSED_FILE,
)

from src.data import (
    load_fixed_partition_data
)

from src.preprocessing import (
    text_quality_summary,
    cwe_leakage_summary,
    preprocess_dataframe,
    validate_preprocessed_text,
)

from src.models import (
    make_word_tfidf,
    make_char_tfidf,
)


ru_df, nonru_df = (
    load_fixed_partition_data()
)

In [ ]:
## Audit Description Quality and Target Leakage

# Missing or empty descriptions are quantified and the raw text is checked for explicit CWE identifiers,
# including direct mentions of the target label.

In [2]:
print("Russian:")
print(
    text_quality_summary(
        ru_df
    )
)

print("\nNon-Russian:")
print(
    text_quality_summary(
        nonru_df
    )
)


print("\nCWE leakage:")

print(
    "RU:",
    cwe_leakage_summary(
        ru_df
    )
)

print(
    "Non-RU:",
    cwe_leakage_summary(
        nonru_df
    )
)

Russian:
{'records': 14451, 'missing_descriptions': 0, 'empty_descriptions': 0}

Non-Russian:
{'records': 41663, 'missing_descriptions': 0, 'empty_descriptions': 0}

CWE leakage:
RU: {'descriptions_with_any_cwe': 0, 'descriptions_with_target_cwe': 0}
Non-RU: {'descriptions_with_any_cwe': 0, 'descriptions_with_target_cwe': 0}


In [ ]:
## Apply Text Normalisation

# The fixed preprocessing pipeline is applied to both domains.
# The resulting text is validated to ensure that preprocessing has not produced empty records
# or left explicit CWE identifiers in the descriptions.

In [4]:
ru_clean = (
    preprocess_dataframe(
        ru_df
    )
)

nonru_clean = (
    preprocess_dataframe(
        nonru_df
    )
)


print(
    validate_preprocessed_text(
        ru_clean
    )
)

print(
    validate_preprocessed_text(
        nonru_clean
    )
)

{'records': 14451, 'empty_after_preprocessing': 0, 'explicit_cwe_ids_remaining': 0}
{'records': 41663, 'empty_after_preprocessing': 0, 'explicit_cwe_ids_remaining': 0}


In [ ]:
## Verify TF-IDF Compatibility

# A small training sample is transformed using both word-level and character-level TF-IDF
# representations as a sanity check before the main classical modelling experiments.

In [5]:
sample = (
    ru_clean.loc[
        ru_clean["split"] == "train",
        TEXT_COL
    ]
    .head(100)
)


word_test = (
    make_word_tfidf()
    .fit_transform(sample)
)

char_test = (
    make_char_tfidf()
    .fit_transform(sample)
)


print(
    "Word:",
    word_test.shape
)

print(
    "Character:",
    char_test.shape
)

Word: (100, 187)
Character: (100, 2256)


In [ ]:
## Export the Preprocessed Data

# The final preprocessed Russian and non-Russian datasets are saved
# while retaining the previously defined partition and leakage-group information.

In [7]:
ru_clean.to_excel(
    RU_PREPROCESSED_FILE,
    index=False
)

nonru_clean.to_excel(
    NONRU_PREPROCESSED_FILE,
    index=False
)


print("Preprocessed datasets saved.")

Preprocessed datasets saved.
